In [3]:
pip install --upgrade statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
INPUT_FILE  = 'analysis_dataset_2021_07.csv'
OUTPUT_FILE = 'ols_results_2021_07.csv'

# Additional files for S11b (age-standardized NB regression)
MORTALITY_ALLAGES_FILE = 'death_2023.csv'       # same file used in script 2 (re-read here for its national row)
MORTALITY_65PLUS_FILE  = 'death_65.csv'         # NCI, Ages 65+ filter, same 2019-2023 window
AGE_POP_FILE           = 'cc-est2024-agesex-all.csv'  # Census Vintage 2024, county x age x sex

# ============================================================
# Step 1: Load data
# ============================================================
print("Loading analysis dataset...")
df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={'FIPS': str})
print(f"Loaded: {len(df):,} counties")

# ============================================================
# Step 2: Prepare variables
# ============================================================

# Exposure: log(density + 1) — handles zero-inflation and right skew
df['log_density_rad']         = np.log(df['density_radiologist'] + 1)
df['log_density_rad_primary'] = np.log(df['density_radiologist_primary'] + 1)
df['log_density_path']        = np.log(df['density_pathologist'] + 1)

# Covariates
df['rucc_2023']                = pd.to_numeric(df['rucc_2023'], errors='coerce')
df['median_household_income']  = pd.to_numeric(df['median_household_income'], errors='coerce')
df['pct_uninsured_adults']     = pd.to_numeric(df['pct_uninsured_adults'], errors='coerce')

# Income in $10,000 units for interpretable coefficient
df['income_10k'] = df['median_household_income'] / 10_000

# Analysis samples
# Main model requires: outcome + all three covariates
df_main = df[
    df['mortality_rate'].notna() &
    df['rucc_2023'].notna() &
    df['income_10k'].notna() &
    df['pct_uninsured_adults'].notna()
].copy()

df_incd = df[
    df['incidence_rate'].notna() &
    df['rucc_2023'].notna() &
    df['income_10k'].notna() &
    df['pct_uninsured_adults'].notna()
].copy()

print(f"\nMain analysis sample (mortality): {len(df_main):,} counties")
print(f"Incidence sample:                  {len(df_incd):,} counties")
print(f"Dropped (suppressed/missing covariates): {len(df) - len(df_main):,}")
print(f"\nExposure range:  log_density_rad {df_main['log_density_rad'].min():.3f} - {df_main['log_density_rad'].max():.3f}")
print(f"RUCC range:      {df_main['rucc_2023'].min():.0f} - {df_main['rucc_2023'].max():.0f}")
print(f"Income range:    ${df_main['median_household_income'].min():,.0f} - ${df_main['median_household_income'].max():,.0f}")
print(f"Uninsured range: {df_main['pct_uninsured_adults'].min():.1f}% - {df_main['pct_uninsured_adults'].max():.1f}%")

# ============================================================
# Step 2b: Analytic-sample descriptive statistics
# These are specific to df_main (the analytic/regression sample), and
# are distinct from the full-dataset descriptive statistics printed in
# script 2 (which run over all counties before analytic-sample
# filtering). Saved here so the manuscript's Results "Descriptive
# Statistics" paragraph is traceable to an actual script output.
# ============================================================
print("\n=== Analytic-Sample Descriptive Statistics (df_main) ===")
print(f"Radiologist density: median = {df_main['density_radiologist'].median():.2f}, "
      f"mean = {df_main['density_radiologist'].mean():.2f}")
n_extreme_main = (df_main['density_radiologist'] == 0).sum()
print(f"Extreme shortage (density = 0): {n_extreme_main:,} counties "
      f"({n_extreme_main/len(df_main)*100:.1f}% of analytic sample)")
mean_rucc = df_main['rucc_2023'].mean()
mean_income = df_main['median_household_income'].mean()
mean_uninsured = df_main['pct_uninsured_adults'].mean()
print(f"Mean RUCC: {mean_rucc:.1f}")
print(f"Mean median household income: ${mean_income:,.0f}")
print(f"Mean proportion uninsured: {mean_uninsured:.1f}%")

descriptive_stats = pd.DataFrame({
    'statistic': ['n_analytic_sample', 'density_median', 'density_mean',
                  'n_extreme_shortage', 'pct_extreme_shortage',
                  'mean_rucc', 'mean_income', 'mean_uninsured_pct',
                  'mortality_rate_mean', 'mortality_rate_sd',
                  'mortality_rate_min', 'mortality_rate_max'],
    'value': [len(df_main),
              round(df_main['density_radiologist'].median(), 4),
              round(df_main['density_radiologist'].mean(), 4),
              n_extreme_main,
              round(n_extreme_main/len(df_main)*100, 1),
              round(mean_rucc, 2),
              round(mean_income, 0),
              round(mean_uninsured, 2),
              round(df_main['mortality_rate'].mean(), 2),
              round(df_main['mortality_rate'].std(), 2),
              round(df_main['mortality_rate'].min(), 2),
              round(df_main['mortality_rate'].max(), 2)]
})
descriptive_stats.to_csv('analytic_sample_descriptive_stats_2021_07.csv', index=False)
print("\nSaved to: analytic_sample_descriptive_stats_2021_07.csv")

# ============================================================
# Step 3: Model 1 — Baseline (no controls)
# ============================================================
print("\n" + "="*60)
print("MODEL 1: Baseline (no controls)")
print("mortality_rate ~ log(density_radiologist + 1)")
print("="*60)

m1 = smf.ols('mortality_rate ~ log_density_rad', data=df_main).fit(
    cov_type='HC3'
)
print(m1.summary2())

# ============================================================
# Step 4: Model 2 — Main model (all three covariates)
# ============================================================
print("\n" + "="*60)
print("MODEL 2: Main model (RUCC + income + uninsured)")
print("mortality_rate ~ log(density_rad+1) + rucc_2023 + income_10k + pct_uninsured_adults")
print("="*60)

m2 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
print(m2.summary2())

# ============================================================
# Step 4b: Model diagnostics for M2 (VIF + residual diagnostics)
# Checks two things not covered by S1-S10: (1) whether the three
# covariates are too collinear for stable coefficient estimation,
# and (2) whether M2's residuals show heteroscedasticity/non-normality
# beyond what HC3 robust SEs are expected to handle.
# ============================================================
print("\n" + "="*60)
print("MODEL 2 DIAGNOSTICS: VIF and residual checks")
print("="*60)

diagnostics_results = []

# --- VIF (collinearity among M2's three covariates) ---
vif_vars = ['rucc_2023', 'income_10k', 'pct_uninsured_adults']
X_vif = sm.add_constant(df_main[vif_vars])
print("\nVariance Inflation Factors (VIF):")
for i, var in enumerate(vif_vars):
    vif_val = variance_inflation_factor(X_vif.values, i + 1)  # +1 skips the constant column
    print(f"  {var}: VIF = {vif_val:.2f}")
    diagnostics_results.append({
        'check': f'VIF_{var}', 'value': round(vif_val, 4),
        'note': 'VIF > 5-10 suggests problematic collinearity'
    })

# --- Residual diagnostics for M2 ---
resid_m2 = m2.resid
fitted_m2 = m2.fittedvalues

# Breusch-Pagan test for heteroscedasticity
bp_stat, bp_pval, _, _ = het_breuschpagan(resid_m2, m2.model.exog)
print(f"\nBreusch-Pagan test (heteroscedasticity): stat = {bp_stat:.4f}, p = {bp_pval:.4f}")
print("  (p < 0.05 indicates heteroscedasticity present; HC3 SEs used throughout are robust to this)")
diagnostics_results.append({
    'check': 'breusch_pagan_pval', 'value': round(bp_pval, 4),
    'note': 'p<0.05 = heteroscedasticity present'
})

# Normality of residuals (skew, kurtosis, Jarque-Bera)
skew_m2 = pd.Series(resid_m2).skew()
kurt_m2 = pd.Series(resid_m2).kurtosis() + 3  # convert to standard (non-excess) kurtosis
jb_stat, jb_pval = sm.stats.stattools.jarque_bera(resid_m2)[:2]
print(f"\nResidual skew: {skew_m2:.3f}  |  kurtosis: {kurt_m2:.3f}  |  Jarque-Bera p = {jb_pval:.4f}")
print(f"  (Large N={len(df_main):,} means OLS estimates remain asymptotically valid even if residuals deviate from normality)")
diagnostics_results.append({
    'check': 'residual_skew', 'value': round(skew_m2, 4), 'note': ''
})
diagnostics_results.append({
    'check': 'residual_kurtosis', 'value': round(kurt_m2, 4), 'note': ''
})
diagnostics_results.append({
    'check': 'jarque_bera_pval', 'value': round(jb_pval, 4),
    'note': 'p<0.05 = residuals deviate from normality'
})

# --- Partial (semi-partial) R^2 of the exposure variable in M2 ---
# Answers: of M2's total R^2 (which jumps sharply from M1's 0.0285 once
# RUCC/income/uninsured are added), how much is uniquely attributable to
# log_density_rad itself, versus to the three covariates? Computed as the
# drop in R^2 when log_density_rad is removed from M2 (a "reduced" model
# with only the three covariates).
m2_reduced = smf.ols(
    'mortality_rate ~ rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
partial_r2_exposure = m2.rsquared - m2_reduced.rsquared
print(f"\nPartial R² of exposure (log_density_rad) in M2:")
print(f"  M2 full R² = {m2.rsquared:.4f}  |  M2 without exposure R² = {m2_reduced.rsquared:.4f}")
print(f"  Exposure's unique contribution to R² = {partial_r2_exposure:.4f}")
print("  (Confirms whether M1→M2's R² jump is driven mainly by the covariates or by the exposure itself)")
diagnostics_results.append({
    'check': 'partial_r2_exposure', 'value': round(partial_r2_exposure, 4),
    'note': 'R2(M2) - R2(M2 without log_density_rad); exposure-specific contribution to fit'
})

diagnostics_df = pd.DataFrame(diagnostics_results)
diagnostics_df.to_csv('model_diagnostics_2021_07.csv', index=False)
print("\nDiagnostics saved to: model_diagnostics_2021_07.csv")

# ============================================================
# Step 5: Key coefficient summary
# ============================================================
print("\n" + "="*60)
print("KEY RESULTS SUMMARY")
print("="*60)

for label, model in [("Model 1 (no controls)", m1),
                      ("Model 2 (full controls)", m2)]:
    coef  = model.params['log_density_rad']
    se    = model.bse['log_density_rad']
    pval  = model.pvalues['log_density_rad']
    ci_lo = model.conf_int().loc['log_density_rad', 0]
    ci_hi = model.conf_int().loc['log_density_rad', 1]
    r2    = model.rsquared
    n     = int(model.nobs)
    print(f"\n{label}")
    print(f"  β (log_density_rad) = {coef:.4f}")
    print(f"  SE = {se:.4f}  |  p = {pval:.4f}")
    print(f"  95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  R² = {r2:.4f}  |  N = {n:,}")

# ============================================================
# Step 6: Sensitivity analyses
# ============================================================
print("\n" + "="*60)
print("SENSITIVITY ANALYSES")
print("="*60)

sensitivity_results = []

# S1: Primary-specialty radiologists only
print("\nS1: Primary-specialty radiologists only")
s1 = smf.ols(
    'mortality_rate ~ log_density_rad_primary + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
coef_s1 = s1.params['log_density_rad_primary']
p_s1    = s1.pvalues['log_density_rad_primary']
ci_s1   = s1.conf_int().loc['log_density_rad_primary']
print(f"  β = {coef_s1:.4f}  |  p = {p_s1:.4f}  |  95% CI: [{ci_s1[0]:.4f}, {ci_s1[1]:.4f}]")
sensitivity_results.append({
    'model':'S1_primary_only', 'exposure':'log_density_rad_primary',
    'outcome':'mortality_rate', 'coef':round(coef_s1,4), 'pval':round(p_s1,4),
    'ci_lo':round(ci_s1[0],4), 'ci_hi':round(ci_s1[1],4), 'n':int(s1.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S2: Pathologist density as alternative exposure
print("\nS2: Pathologist density (robustness)")
s2 = smf.ols(
    'mortality_rate ~ log_density_path + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
coef_s2 = s2.params['log_density_path']
p_s2    = s2.pvalues['log_density_path']
ci_s2   = s2.conf_int().loc['log_density_path']
print(f"  β = {coef_s2:.4f}  |  p = {p_s2:.4f}  |  95% CI: [{ci_s2[0]:.4f}, {ci_s2[1]:.4f}]")
sensitivity_results.append({
    'model':'S2_pathologist', 'exposure':'log_density_path',
    'outcome':'mortality_rate', 'coef':round(coef_s2,4), 'pval':round(p_s2,4),
    'ci_lo':round(ci_s2[0],4), 'ci_hi':round(ci_s2[1],4), 'n':int(s2.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S3: Incidence as outcome
print("\nS3: Incidence as outcome")
s3 = smf.ols(
    'incidence_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_incd
).fit(cov_type='HC3')
coef_s3 = s3.params['log_density_rad']
p_s3    = s3.pvalues['log_density_rad']
ci_s3   = s3.conf_int().loc['log_density_rad']
print(f"  β = {coef_s3:.4f}  |  p = {p_s3:.4f}  |  95% CI: [{ci_s3[0]:.4f}, {ci_s3[1]:.4f}]")
sensitivity_results.append({
    'model':'S3_incidence_outcome', 'exposure':'log_density_rad',
    'outcome':'incidence_rate', 'coef':round(coef_s3,4), 'pval':round(p_s3,4),
    'ci_lo':round(ci_s3[0],4), 'ci_hi':round(ci_s3[1],4), 'n':int(s3.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S4: Exclude top 1% density counties (academic medical center outliers)
print("\nS4: Exclude top 1% density outliers (academic medical centers)")
p99 = df_main['density_radiologist'].quantile(0.99)
df_no_outlier = df_main[df_main['density_radiologist'] <= p99].copy()
s4 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_no_outlier
).fit(cov_type='HC3')
coef_s4 = s4.params['log_density_rad']
p_s4    = s4.pvalues['log_density_rad']
ci_s4   = s4.conf_int().loc['log_density_rad']
n_excl  = len(df_main) - len(df_no_outlier)
print(f"  Top 1% threshold: {p99:.2f}/100k  |  Excluded: {n_excl:,} counties")
print(f"  β = {coef_s4:.4f}  |  p = {p_s4:.4f}  |  95% CI: [{ci_s4[0]:.4f}, {ci_s4[1]:.4f}]")
sensitivity_results.append({
    'model':'S4_no_outliers', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s4,4), 'pval':round(p_s4,4),
    'ci_lo':round(ci_s4[0],4), 'ci_hi':round(ci_s4[1],4), 'n':int(s4.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S5: Rural counties only (RUCC 4-9)
print("\nS5: Rural counties only (RUCC 4-9)")
df_rural = df_main[df_main['rucc_2023'] >= 4].copy()
s5 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_rural
).fit(cov_type='HC3')
coef_s5 = s5.params['log_density_rad']
p_s5    = s5.pvalues['log_density_rad']
ci_s5   = s5.conf_int().loc['log_density_rad']
print(f"  Rural counties: {len(df_rural):,}")
print(f"  β = {coef_s5:.4f}  |  p = {p_s5:.4f}  |  95% CI: [{ci_s5[0]:.4f}, {ci_s5[1]:.4f}]")
sensitivity_results.append({
    'model':'S5_rural_only', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s5,4), 'pval':round(p_s5,4),
    'ci_lo':round(ci_s5[0],4), 'ci_hi':round(ci_s5[1],4), 'n':int(s5.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S6: RUCC-only controls (drop income + uninsured) — tests robustness to covariate choice
print("\nS6: RUCC-only controls (drop income + uninsured)")
s6 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023',
    data=df_main
).fit(cov_type='HC3')
coef_s6 = s6.params['log_density_rad']
p_s6    = s6.pvalues['log_density_rad']
ci_s6   = s6.conf_int().loc['log_density_rad']
print(f"  β = {coef_s6:.4f}  |  p = {p_s6:.4f}  |  95% CI: [{ci_s6[0]:.4f}, {ci_s6[1]:.4f}]")
sensitivity_results.append({
    'model':'S6_rucc_only', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s6,4), 'pval':round(p_s6,4),
    'ci_lo':round(ci_s6[0],4), 'ci_hi':round(ci_s6[1],4), 'n':int(s6.nobs),
    'controls':'rucc_2023'
})

# S7: 2026 NPPES workforce snapshot (same 2021 population denominator) —
# tests whether the core finding holds using a more recent workforce
# snapshot, while holding the population/outcome side fixed at 2021.
print("\nS7: 2026 NPPES workforce snapshot (sensitivity to workforce data vintage)")
NPPES_2026_FILE = 'analysis_dataset_2026_fixed.csv'
df_2026 = pd.read_csv(NPPES_2026_FILE, low_memory=False, dtype={'FIPS': str})

df_2026['log_density_rad'] = np.log(df_2026['density_radiologist'] + 1)
df_2026['rucc_2023'] = pd.to_numeric(df_2026['rucc_2023'], errors='coerce')
df_2026['median_household_income'] = pd.to_numeric(df_2026['median_household_income'], errors='coerce')
df_2026['pct_uninsured_adults'] = pd.to_numeric(df_2026['pct_uninsured_adults'], errors='coerce')
df_2026['income_10k'] = df_2026['median_household_income'] / 10_000

df_main_2026 = df_2026[
    df_2026['mortality_rate'].notna() &
    df_2026['rucc_2023'].notna() &
    df_2026['income_10k'].notna() &
    df_2026['pct_uninsured_adults'].notna()
].copy()

s7 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main_2026
).fit(cov_type='HC3')
coef_s7 = s7.params['log_density_rad']
p_s7    = s7.pvalues['log_density_rad']
ci_s7   = s7.conf_int().loc['log_density_rad']
print(f"  N = {len(df_main_2026):,} counties (2026 NPPES snapshot)")
print(f"  β = {coef_s7:.4f}  |  p = {p_s7:.4f}  |  95% CI: [{ci_s7[0]:.4f}, {ci_s7[1]:.4f}]")
sensitivity_results.append({
    'model':'S7_2026_nppes_snapshot', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s7,4), 'pval':round(p_s7,4),
    'ci_lo':round(ci_s7[0],4), 'ci_hi':round(ci_s7[1],4), 'n':int(s7.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# ============================================================
# Post-hoc exploration of the null finding in S5 (S8-S10)
# S5 found a null/reversed association within rural counties (RUCC>=4).
# The following three analyses were added after observing that result,
# to probe whether it reflects genuine effect heterogeneity or is an
# artifact of sample size / cutoff choice / outliers within df_rural.
# ============================================================

# S8: Split df_rural by USDA's own documented urban-population tiers
# (RUCC 4-5 = nonmetro w/ urban pop >=20,000; RUCC 6-9 = smaller/more
# remote nonmetro), per USDA ERS RUCC documentation. Tests whether the
# S5 null finding is uniform across nonmetro counties or concentrated
# in the more remote tier.
print("\nS8: Rural sub-tiers (RUCC 4-5 vs RUCC 6-9, per USDA urban-population tiers)")
df_rural_45 = df_rural[df_rural['rucc_2023'].isin([4, 5])].copy()
df_rural_69 = df_rural[df_rural['rucc_2023'].isin([6, 7, 8, 9])].copy()

for label, sub_df in [('S8_rucc_4_5', df_rural_45), ('S8_rucc_6_9', df_rural_69)]:
    m = smf.ols(
        'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
        data=sub_df
    ).fit(cov_type='HC3')
    coef = m.params['log_density_rad']
    p    = m.pvalues['log_density_rad']
    ci   = m.conf_int().loc['log_density_rad']
    print(f"  [{label}] N = {len(sub_df):,}  |  β = {coef:.4f}  |  p = {p:.4f}  |  95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
    sensitivity_results.append({
        'model':label, 'exposure':'log_density_rad',
        'outcome':'mortality_rate', 'coef':round(coef,4), 'pval':round(p,4),
        'ci_lo':round(ci[0],4), 'ci_hi':round(ci[1],4), 'n':int(m.nobs),
        'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
    })

# S9: Interaction model on the full sample (no sample-splitting, so no
# loss of power). Tests directly whether the density-mortality
# association differs by rural status, using a binary rural indicator
# (RUCC>=4) interacted with log_density_rad in place of continuous RUCC.
print("\nS9: Density x rural-status interaction (full sample, no sample split)")
df_main['rural_indicator'] = (df_main['rucc_2023'] >= 4).astype(int)
s9 = smf.ols(
    'mortality_rate ~ log_density_rad * rural_indicator + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
coef_s9 = s9.params['log_density_rad:rural_indicator']
p_s9    = s9.pvalues['log_density_rad:rural_indicator']
ci_s9   = s9.conf_int().loc['log_density_rad:rural_indicator']
print(f"  N = {len(df_main):,}  |  interaction β = {coef_s9:.4f}  |  p = {p_s9:.4f}  |  95% CI: [{ci_s9[0]:.4f}, {ci_s9[1]:.4f}]")
sensitivity_results.append({
    'model':'S9_density_x_rural_interaction', 'exposure':'log_density_rad:rural_indicator',
    'outcome':'mortality_rate', 'coef':round(coef_s9,4), 'pval':round(p_s9,4),
    'ci_lo':round(ci_s9[0],4), 'ci_hi':round(ci_s9[1],4), 'n':int(s9.nobs),
    'controls':'rural_indicator + income_10k + pct_uninsured_adults (main effects) + interaction'
})

# S10: Outlier check within the S5 rural subsample itself (distinct from
# S4, which only checked outliers in the full sample). Drops the top 1%
# of density_radiologist within df_rural and reruns the S5 model.
print("\nS10: Outlier check within rural subsample (top 1% of df_rural density)")
p99_rural = df_rural['density_radiologist'].quantile(0.99)
df_rural_no_outlier = df_rural[df_rural['density_radiologist'] <= p99_rural].copy()
s10 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_rural_no_outlier
).fit(cov_type='HC3')
coef_s10 = s10.params['log_density_rad']
p_s10    = s10.pvalues['log_density_rad']
ci_s10   = s10.conf_int().loc['log_density_rad']
print(f"  Top 1% threshold (within rural): {p99_rural:.2f}/100k  |  Excluded: {len(df_rural) - len(df_rural_no_outlier)} counties")
print(f"  β = {coef_s10:.4f}  |  p = {p_s10:.4f}  |  95% CI: [{ci_s10[0]:.4f}, {ci_s10[1]:.4f}]")
sensitivity_results.append({
    'model':'S10_rural_no_outliers', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s10,4), 'pval':round(p_s10,4),
    'ci_lo':round(ci_s10[0],4), 'ci_hi':round(ci_s10[1],4), 'n':int(s10.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S11: Negative binomial regression on raw mortality counts, with
# population as an offset, instead of modeling the pre-computed rate
# linearly. This is the methodologically preferred approach for count
# outcomes (mortality_rate is a derived rate; mortality_avg_annual_count
# is the underlying count). Tests whether M2's conclusion holds under a
# model family that matches the data-generating process of count data.
# Poisson is fit first only to report the dispersion statistic (a value
# well above 1 indicates overdispersion, i.e. NB is the more appropriate
# choice than Poisson); NB is reported as the S11 result either way.
print("\nS11: Negative binomial regression (raw mortality counts + population offset)")
df_main['mortality_avg_annual_count'] = pd.to_numeric(
    df_main['mortality_avg_annual_count'], errors='coerce'
)
df_nb = df_main[df_main['mortality_avg_annual_count'].notna()].copy()
df_nb['log_population'] = np.log(df_nb['population_2021'])

# Poisson fit, for reporting the dispersion statistic only
poisson_fit = sm.GLM(
    df_nb['mortality_avg_annual_count'],
    sm.add_constant(df_nb[['log_density_rad', 'rucc_2023', 'income_10k', 'pct_uninsured_adults']]),
    offset=df_nb['log_population'],
    family=sm.families.Poisson()
).fit()
dispersion = poisson_fit.pearson_chi2 / poisson_fit.df_resid
print(f"  Poisson dispersion statistic: {dispersion:.2f} (>>1 indicates overdispersion; NB used below regardless)")

diagnostics_results.append({
    'check': 'poisson_dispersion_statistic_S11', 'value': round(dispersion, 4),
    'note': 'Pearson chi2/df from a Poisson fit on mortality_avg_annual_count; '
            '>>1 indicates overdispersion, justifying negative binomial (used in S11) over Poisson'
})
pd.DataFrame(diagnostics_results).to_csv('model_diagnostics_2021_07.csv', index=False)
print("  (Poisson dispersion statistic appended to model_diagnostics_2021_07.csv)")

nb_fit = sm.GLM(
    df_nb['mortality_avg_annual_count'],
    sm.add_constant(df_nb[['log_density_rad', 'rucc_2023', 'income_10k', 'pct_uninsured_adults']]),
    offset=df_nb['log_population'],
    family=sm.families.NegativeBinomial()
).fit()

coef_s11 = nb_fit.params['log_density_rad']
p_s11    = nb_fit.pvalues['log_density_rad']
ci_s11   = nb_fit.conf_int().loc['log_density_rad']
irr_s11  = np.exp(coef_s11)
print(f"  N = {len(df_nb):,}  |  log-IRR = {coef_s11:.4f} (IRR = {irr_s11:.4f})  |  p = {p_s11:.4f}")
print(f"  95% CI (log-IRR): [{ci_s11[0]:.4f}, {ci_s11[1]:.4f}]")
print("  Note: this coefficient is a log-rate-ratio (multiplicative), not directly comparable in scale to M2's OLS beta.")
sensitivity_results.append({
    'model':'S11_negative_binomial', 'exposure':'log_density_rad',
    'outcome':'mortality_avg_annual_count (offset=log(population_2021))',
    'coef':round(coef_s11,4), 'pval':round(p_s11,4),
    'ci_lo':round(ci_s11[0],4), 'ci_hi':round(ci_s11[1],4), 'n':int(len(df_nb)),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S11b: Same negative binomial model as S11, but with the offset
# replaced by "expected deaths" (indirect age standardization), instead
# of raw total population. This addresses the concern that S11's
# offset (population_2021) does not account for county age structure,
# unlike M2's outcome (mortality_rate), which NCI already age-adjusts.
#
# expected_deaths_county = national_rate_65plus * county_pop_65plus
#                         + national_rate_under65 * county_pop_under65
#
# National age-specific rates are derived from NCI's official national
# totals (FIPS=00000 rows) for All-Ages and Ages-65+ mortality reports
# (same 2019-2023 window), divided by national 65+/<65 population
# (summed from the Census county x age file). County-level 65+/<65
# population comes from the same Census file.
print("\nS11b: Negative binomial regression, age-standardized offset (expected deaths)")

# National age-specific rates
mort_allages_raw = pd.read_csv(MORTALITY_ALLAGES_FILE, skiprows=8, dtype=str)
nat_allages_count = pd.to_numeric(
    mort_allages_raw.loc[mort_allages_raw['FIPS'] == '00000', 'Average Annual Count'].iloc[0]
)
mort_65plus_raw = pd.read_csv(MORTALITY_65PLUS_FILE, skiprows=8, dtype=str)
nat_65plus_count = pd.to_numeric(
    mort_65plus_raw.loc[mort_65plus_raw['FIPS'] == '00000', 'Average Annual Count'].iloc[0]
)
nat_under65_count = nat_allages_count - nat_65plus_count

age_pop_raw = pd.read_csv(AGE_POP_FILE, encoding='latin-1',
                          usecols=['SUMLEV', 'STATE', 'COUNTY', 'YEAR', 'POPESTIMATE', 'AGE65PLUS_TOT'])
age_pop = age_pop_raw[(age_pop_raw['SUMLEV'] == 50) & (age_pop_raw['YEAR'] == 3)].copy()  # YEAR=3 -> July 1, 2021
age_pop['FIPS'] = (age_pop['STATE'].astype(str).str.zfill(2) + age_pop['COUNTY'].astype(str).str.zfill(3))
age_pop['pop_65plus']  = age_pop['AGE65PLUS_TOT']
age_pop['pop_under65'] = age_pop['POPESTIMATE'] - age_pop['AGE65PLUS_TOT']

nat_pop_65plus  = age_pop['pop_65plus'].sum()
nat_pop_under65 = age_pop['pop_under65'].sum()

nat_rate_65plus  = nat_65plus_count  / nat_pop_65plus  * 100_000
nat_rate_under65 = nat_under65_count / nat_pop_under65 * 100_000
print(f"  National rate, ages 65+:  {nat_rate_65plus:.2f} per 100,000")
print(f"  National rate, ages <65:  {nat_rate_under65:.2f} per 100,000")

df_nb_std = df_nb.merge(age_pop[['FIPS', 'pop_65plus', 'pop_under65']], on='FIPS', how='left')
df_nb_std = df_nb_std[df_nb_std['pop_65plus'].notna()].copy()

df_nb_std['expected_deaths'] = (
    nat_rate_65plus  / 100_000 * df_nb_std['pop_65plus'] +
    nat_rate_under65 / 100_000 * df_nb_std['pop_under65']
)
df_nb_std['log_expected_deaths'] = np.log(df_nb_std['expected_deaths'])

nb_std_fit = sm.GLM(
    df_nb_std['mortality_avg_annual_count'],
    sm.add_constant(df_nb_std[['log_density_rad', 'rucc_2023', 'income_10k', 'pct_uninsured_adults']]),
    offset=df_nb_std['log_expected_deaths'],
    family=sm.families.NegativeBinomial()
).fit()

coef_s11b = nb_std_fit.params['log_density_rad']
p_s11b    = nb_std_fit.pvalues['log_density_rad']
ci_s11b   = nb_std_fit.conf_int().loc['log_density_rad']
irr_s11b  = np.exp(coef_s11b)
print(f"  N = {len(df_nb_std):,}  |  log-IRR = {coef_s11b:.4f} (IRR = {irr_s11b:.4f})  |  p = {p_s11b:.4f}")
print(f"  95% CI (log-IRR): [{ci_s11b[0]:.4f}, {ci_s11b[1]:.4f}]")
print(f"  vs. S11 (unadjusted population offset): log-IRR = {coef_s11:.4f}, p = {p_s11:.4f}")
sensitivity_results.append({
    'model':'S11b_negative_binomial_age_standardized', 'exposure':'log_density_rad',
    'outcome':'mortality_avg_annual_count (offset=log(expected_deaths))',
    'coef':round(coef_s11b,4), 'pval':round(p_s11b,4),
    'ci_lo':round(ci_s11b[0],4), 'ci_hi':round(ci_s11b[1],4), 'n':int(len(df_nb_std)),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# ============================================================
# Step 7: Save results
# ============================================================
results = []

for label, model, var, outcome, controls in [
    ('M1_baseline', m1, 'log_density_rad', 'mortality_rate', 'none'),
    ('M2_main',     m2, 'log_density_rad', 'mortality_rate',
     'rucc_2023 + income_10k + pct_uninsured_adults'),
]:
    ci = model.conf_int().loc[var]
    results.append({
        'model':    label,
        'exposure': var,
        'outcome':  outcome,
        'coef':     round(model.params[var], 4),
        'se':       round(model.bse[var], 4),
        'pval':     round(model.pvalues[var], 4),
        'ci_lo':    round(ci[0], 4),
        'ci_hi':    round(ci[1], 4),
        'r2':       round(model.rsquared, 4),
        'n':        int(model.nobs),
        'controls': controls
    })

results += sensitivity_results
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n\nAll results saved to: {OUTPUT_FILE}")
print("\n" + "="*60)
print("DONE. Check M2 β before proceeding to Script 4 (AI scenarios).")
print("="*60)

Loading analysis dataset...
Loaded: 3,135 counties

Main analysis sample (mortality): 3,073 counties
Incidence sample:                  3,018 counties
Dropped (suppressed/missing covariates): 62

Exposure range:  log_density_rad 0.000 - 5.974
RUCC range:      1 - 9
Income range:    $28,236 - $153,716
Uninsured range: 2.6% - 44.5%

=== Analytic-Sample Descriptive Statistics (df_main) ===
Radiologist density: median = 0.33, mean = 5.29
Extreme shortage (density = 0): 1,334 counties (43.4% of analytic sample)
Mean RUCC: 5.2
Mean median household income: $58,849
Mean proportion uninsured: 13.9%

Saved to: analytic_sample_descriptive_stats_2021_07.csv

MODEL 1: Baseline (no controls)
mortality_rate ~ log(density_radiologist + 1)
                  Results: Ordinary least squares
Model:              OLS              Adj. R-squared:     0.028     
Dependent Variable: mortality_rate   AIC:                29172.0224
Date:               2026-08-24 17:52 BIC:                29184.0832
No. Observat